In [ ]:
from cmath import nan
from json.decoder import NaN

import pandas as pd

df = pd.read_csv("Consp2VecD-Final_Dataset/conspiracy.csv", index_col=0)

df = df.reset_index().rename(columns={"index": "type"})

cols = df.columns[df.apply(lambda col: col.astype(str).str.contains("Commento su:").any())]

df[cols] = df[cols].apply(
    lambda col: col.astype(str).str.replace("Commento su:", "", regex=False)
)

import numpy as np

# 0️⃣ se type è index → portalo a colonna (safe)
if "type" not in df.columns:
    df = df.reset_index(names="type")
else:
    df = df.reset_index(drop=True)

# 1️⃣ assicuriamoci che l'id del post si chiami ID_Post
df = df.rename(columns={"post_id": "ID_Post"})

# 2️⃣ contatore commenti che riparte per ogni ID_Post
df["comment_counter"] = (
    df[df["type"] == "comment"]
    .groupby("ID_Post")
    .cumcount() + 1
)

# 3️⃣ costruzione ID finale
df["id"] = df["ID_Post"].astype(str)

mask_comment = df["type"] == "comment"
df.loc[mask_comment, "id"] = (
    df.loc[mask_comment, "ID_Post"].astype(str)
    + "_"
    + df.loc[mask_comment, "comment_counter"].astype(int).astype(str)
)

# 4️⃣ pulizia
df = df.drop(columns=["comment_counter"])

df = df.drop(columns=["ID_Post", "flair", "score", "category"])

df.loc[df["type"] == "comment", "title"] = None

mask_short = (
    (df["type"] == "comment")
    & (df["body"].fillna("").astype(str).str.split().str.len() < 5)
)

df = df[~mask_short].copy()


df.to_csv("Consp2VecD-Final_Dataset/conspiracy_cleaned.csv", index=False, encoding="utf-8")


In [ ]:
import pandas as pd
df = pd.read_csv("Consp2VecD-Final_Dataset/non_conspiracy.csv", index_col=0)

df = df.reset_index().rename(columns={"index": "type"})

cols = df.columns[df.apply(lambda col: col.astype(str).str.contains("Commento su:").any())]

df[cols] = df[cols].apply(
    lambda col: col.astype(str).str.replace("Commento su:", "", regex=False)
)

# 0️⃣ se type è index → portalo a colonna (safe)
if "type" not in df.columns:
    df = df.reset_index(names="type")
else:
    df = df.reset_index(drop=True)

# 1️⃣ assicuriamoci che l'id del post si chiami ID_Post
df = df.rename(columns={"post_id": "ID_Post"})

# 2️⃣ contatore commenti che riparte per ogni ID_Post
df["comment_counter"] = (
    df[df["type"] == "comment"]
    .groupby("ID_Post")
    .cumcount() + 1
)

# 3️⃣ costruzione ID finale
df["id"] = df["ID_Post"].astype(str)

mask_comment = df["type"] == "comment"
df.loc[mask_comment, "id"] = (
    df.loc[mask_comment, "ID_Post"].astype(str)
    + "_"
    + df.loc[mask_comment, "comment_counter"].astype(int).astype(str)
)

# 4️⃣ pulizia
df = df.drop(columns=["comment_counter"])

df = df.drop(columns=["ID_Post", "flair", "score", "category"])

df.loc[df["type"] == "comment", "title"] = None

mask_short = (
    (df["type"] == "comment")
    & (df["body"].fillna("").astype(str).str.split().str.len() < 5)
)

df = df[~mask_short].copy()

df.to_csv("Consp2VecD-Final_Dataset/non_conspiracy_cleaned.csv", index=False, encoding="utf-8")


In [ ]:

# df: DataFrame con colonne ['type','title','author','body','created_utc','subreddit_name','id']
def cut_date(df):
    d = df.copy()

    # 1) Parse date (created_utc può essere epoch seconds oppure datetime string)
    if np.issubdtype(d["created_utc"].dtype, np.number):
        d["created_dt"] = pd.to_datetime(d["created_utc"], unit="s", utc=True, errors="coerce")
    else:
        d["created_dt"] = pd.to_datetime(d["created_utc"], utc=True, errors="coerce")

    cutoff = pd.Timestamp("2025-01-01", tz="UTC")

    # 2) Definisci l'id "base" del post: per i commenti togli il suffisso _x
    #    (es. abc123_4 -> abc123)
    d["base_id"] = d["id"].astype(str).str.split("_", n=1).str[0]

    # 3) Trova i post antecedenti a gennaio 2025 (da eliminare)
    old_posts_base_ids = d.loc[
        (d["type"].astype(str).str.lower() == "post") & (d["created_dt"] < cutoff),
        "base_id"
    ].unique()

    # 4) Elimina:
    #    - i post vecchi
    #    - TUTTI i commenti associati (stesso base_id)
    mask_drop = d["base_id"].isin(old_posts_base_ids)

    filtered_df = d.loc[~mask_drop].drop(columns=["created_dt", "base_id"]).reset_index(drop=True)

    return filtered_df


In [ ]:
conspiracy = pd.read_csv("Consp2VecD-Final_Dataset/conspiracy_cleaned.csv", index_col=0)
non_conspiracy = pd.read_csv("Consp2VecD-Final_Dataset/non_conspiracy_cleaned.csv", index_col=0)

conspiracy.reset_index(drop=True, inplace=True)
non_conspiracy.reset_index(drop=True, inplace=True)

In [ ]:
conspiracy

In [ ]:
conspiracy = cut_date(conspiracy)
non_conspiracy = cut_date(non_conspiracy)


conspiracy["n_type"] = conspiracy.index
conspiracy = conspiracy.reset_index(drop=True)
conspiracy["type"] = conspiracy["n_type"]
conspiracy.drop(columns=["n_type"], inplace=True)

non_conspiracy["n_type"] = non_conspiracy.index
non_conspiracy = non_conspiracy.reset_index(drop=True)
non_conspiracy["type"] = non_conspiracy["n_type"]
non_conspiracy.drop(columns=["n_type"], inplace=True)



conspiracy.to_csv("Consp2VecD-Final_Dataset/conspiracy_cleaned.csv", index=False, encoding="utf-8")
non_conspiracy.to_csv("Consp2VecD-Final_Dataset/non_conspiracy_cleaned.csv", index=False, encoding="utf-8")


In [ ]:
conspiracy

In [ ]:
non_conspiracy

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import locale

START_DATE = pd.Timestamp("2025-01-01", tz="UTC")  # da gennaio 2025
FREQ = "M"      # "M" mese, "W" settimana, "D" giorno
TOP_N = 10      # numero subreddit principali da mostrare

# ---------------------------
# Robust datetime parsing
# ---------------------------
def add_created_dt(df: pd.DataFrame, col="created_utc") -> pd.DataFrame:
    out = df.copy()

    s = pd.to_numeric(out[col], errors="coerce")
    dt_s = pd.to_datetime(s, unit="s", utc=True, errors="coerce")

    if dt_s.notna().mean() < 0.5:
        dt_ms = pd.to_datetime(s, unit="ms", utc=True, errors="coerce")
        out["created_dt"] = dt_ms if dt_ms.notna().mean() > dt_s.notna().mean() else dt_s
    else:
        out["created_dt"] = dt_s

    if out["created_dt"].notna().mean() < 0.5:
        dt_str = pd.to_datetime(out[col], utc=True, errors="coerce")
        if dt_str.notna().mean() > out["created_dt"].notna().mean():
            out["created_dt"] = dt_str

    out = out.dropna(subset=["created_dt"]).sort_values("created_dt").set_index("created_dt")
    return out

# ---------------------------
# Split posts/comments
# ---------------------------
def split_posts_comments(conspiracy: pd.DataFrame):
    if getattr(conspiracy.index, "name", None) == "type":
        conspiracy = conspiracy.reset_index()

    req = {"type", "subreddit_name", "created_utc"}
    missing = req - set(conspiracy.columns)
    if missing:
        raise ValueError(f"Colonne mancanti: {missing}")

    df = conspiracy.copy()
    df["type_norm"] = df["type"].astype(str).str.lower().str.strip()

    posts_df = df[df["type_norm"].isin(["post", "submission"])].copy()
    comments_df = df[df["type_norm"].isin(["comment"])].copy()
    return posts_df, comments_df

# ---------------------------
# Build time x subreddit matrix (top N + Other), filtered from START_DATE
# ---------------------------
def time_by_subreddit(df: pd.DataFrame, start_date=START_DATE, freq="M", top_n=10) -> pd.DataFrame:
    d = add_created_dt(df, col="created_utc")

    # filtro temporale
    d = d.loc[d.index >= start_date].copy()

    d["subreddit_name"] = d["subreddit_name"].astype(str).str.strip()

    if d.empty:
        return pd.DataFrame()

    top = d["subreddit_name"].value_counts().head(top_n).index
    d["subreddit_grp"] = d["subreddit_name"].where(d["subreddit_name"].isin(top), other="Other")

    g = (
        d.groupby([pd.Grouper(freq=freq), "subreddit_grp"])
         .size()
         .unstack(fill_value=0)
         .sort_index()
    )

    # ordina colonne per totale desc
    g = g[g.sum().sort_values(ascending=False).index]
    return g

# ---------------------------
# Plot stacked area
# ---------------------------
def plot_stacked_area(g: pd.DataFrame, title: str):
    try:
        locale.setlocale(locale.LC_TIME, "en_US.UTF-8")
    except locale.Error:
        pass

    fig, ax = plt.subplots(figsize=(13, 6))

    ax.stackplot(g.index, [g[c].values for c in g.columns], labels=list(g.columns))
    ax.set_title(title)
    ax.set_xlabel("Month-Year (UTC)")
    ax.set_ylabel("Count")

    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b-%Y"))
    fig.autofmt_xdate(rotation=45)

    ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)
    plt.tight_layout()
    plt.show()

# ===========================
# RUN
# ===========================
posts_df, comments_df = split_posts_comments(conspiracy)

posts_g = time_by_subreddit(posts_df, start_date=START_DATE, freq=FREQ, top_n=TOP_N)
comments_g = time_by_subreddit(comments_df, start_date=START_DATE, freq=FREQ, top_n=TOP_N)

if posts_g.empty:
    raise ValueError("Nessun POST dal 2025-01-01 (o parsing created_utc fallito / type non corrisponde).")
if comments_g.empty:
    raise ValueError("Nessun COMMENTO dal 2025-01-01 (o parsing created_utc fallito / type non corrisponde).")

plot_stacked_area(posts_g, f"Posts over time by subreddit (Top {TOP_N} + Other) — from Jan 2025")
plot_stacked_area(comments_g, f"Comments over time by subreddit (Top {TOP_N} + Other) — from Jan 2025")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import locale

START_DATE = pd.Timestamp("2025-01-01", tz="UTC")
FREQ = "MS"  # Month Start: più comodo per avere mesi "puliti" (2025-01-01, 2025-02-01, ...)

# ---------------------------
# Robust datetime parsing
# ---------------------------
def add_created_dt(df: pd.DataFrame, col="created_utc") -> pd.DataFrame:
    out = df.copy()
    s = pd.to_numeric(out[col], errors="coerce")

    dt_s = pd.to_datetime(s, unit="s", utc=True, errors="coerce")
    if dt_s.notna().mean() < 0.5:
        dt_ms = pd.to_datetime(s, unit="ms", utc=True, errors="coerce")
        out["created_dt"] = dt_ms if dt_ms.notna().mean() > dt_s.notna().mean() else dt_s
    else:
        out["created_dt"] = dt_s

    if out["created_dt"].notna().mean() < 0.5:
        dt_str = pd.to_datetime(out[col], utc=True, errors="coerce")
        if dt_str.notna().mean() > out["created_dt"].notna().mean():
            out["created_dt"] = dt_str

    out = out.dropna(subset=["created_dt"]).sort_values("created_dt").set_index("created_dt")
    return out

def split_posts_comments(conspiracy: pd.DataFrame):
    if getattr(conspiracy.index, "name", None) == "type":
        conspiracy = conspiracy.reset_index()

    req = {"type", "subreddit_name", "created_utc"}
    missing = req - set(conspiracy.columns)
    if missing:
        raise ValueError(f"Colonne mancanti: {missing}")

    df = conspiracy.copy()
    df["type_norm"] = df["type"].astype(str).str.lower().str.strip()

    posts_df = df[df["type_norm"].isin(["post", "submission"])].copy()
    comments_df = df[df["type_norm"].isin(["comment"])].copy()

    # se qui uno dei due è 0, probabilmente type ha valori diversi
    # print(df["type_norm"].value_counts().head(20))

    return posts_df, comments_df

def make_month_subreddit_matrix(df: pd.DataFrame, start_date=START_DATE, freq=FREQ) -> pd.DataFrame:
    d = add_created_dt(df, col="created_utc")
    d = d.loc[d.index >= start_date].copy()

    d["subreddit_name"] = d["subreddit_name"].astype(str).str.strip()
    if d.empty:
        return pd.DataFrame()

    mat = (
        d.groupby([pd.Grouper(freq=freq), "subreddit_name"])
         .size()
         .unstack(fill_value=0)
         .sort_index()
    )
    return mat

def align_matrices(posts_mat: pd.DataFrame, comments_mat: pd.DataFrame, start_date=START_DATE):
    # 1) Unione di tutti i subreddit
    all_subs = posts_mat.columns.union(comments_mat.columns)

    # 2) Ordine unico: per volume complessivo (post+commenti)
    totals = posts_mat.reindex(columns=all_subs, fill_value=0).sum(axis=0) + \
             comments_mat.reindex(columns=all_subs, fill_value=0).sum(axis=0)
    subs_order = totals.sort_values(ascending=False).index

    # 3) Intervallo mesi completo (così non “mancano gruppi”/righe)
    max_idx = None
    if not posts_mat.empty:
        max_idx = posts_mat.index.max() if max_idx is None else max(max_idx, posts_mat.index.max())
    if not comments_mat.empty:
        max_idx = comments_mat.index.max() if max_idx is None else max(max_idx, comments_mat.index.max())

    if max_idx is None:
        raise ValueError("Entrambe le matrici sono vuote dopo il filtro temporale.")

    full_months = pd.date_range(start=start_date.floor("D"), end=max_idx, freq=FREQ, tz="UTC")

    # 4) Reindex: stesso asse temporale + stesse colonne, riempi 0
    posts_aligned = posts_mat.reindex(index=full_months, columns=subs_order, fill_value=0)
    comments_aligned = comments_mat.reindex(index=full_months, columns=subs_order, fill_value=0)

    return posts_aligned, comments_aligned

def plot_heatmap(mat: pd.DataFrame, title: str, max_xticks: int = 30):
    # mesi in inglese
    try:
        locale.setlocale(locale.LC_TIME, "en_US.UTF-8")
    except locale.Error:
        pass

    n_rows, n_cols = mat.shape
    fig_w = max(12, min(0.25 * n_cols, 60))
    fig_h = max(5, min(0.35 * n_rows, 25))

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    im = ax.imshow(mat.values, aspect="auto")

    ax.set_title(title)
    ax.set_xlabel("Subreddit")
    ax.set_ylabel("Month-Year (UTC)")

    y_labels = [idx.strftime("%b-%Y") for idx in mat.index.to_pydatetime()]
    ax.set_yticks(range(len(y_labels)))
    ax.set_yticklabels(y_labels)

    # tick X: tutti se pochi, altrimenti diradati
    if n_cols <= max_xticks:
        ax.set_xticks(range(n_cols))
        ax.set_xticklabels(mat.columns, rotation=90)
    else:
        step = max(1, n_cols // max_xticks)
        xt = list(range(0, n_cols, step))
        ax.set_xticks(xt)
        ax.set_xticklabels([mat.columns[i] for i in xt], rotation=90)

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Count")

    plt.tight_layout()
    plt.show()

# ===========================
# RUN
# ===========================
posts_df, comments_df = split_posts_comments(conspiracy)

posts_mat = make_month_subreddit_matrix(posts_df)
comments_mat = make_month_subreddit_matrix(comments_df)

# Allineamento: stesso ordine subreddit + mesi completi
posts_mat, comments_mat = align_matrices(posts_mat, comments_mat, start_date=START_DATE)

plot_heatmap(posts_mat, "POSTS — monthly volume by subreddit (aligned) — from Jan 2025")
plot_heatmap(comments_mat, "COMMENTS — monthly volume by subreddit (aligned) — from Jan 2025")


In [ ]:
import numpy as np

def subreddit_stats(df: pd.DataFrame) -> pd.DataFrame:
    required = {'title','author','body','created_utc','subreddit_name','id','type'}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    # Copia per non modificare df originale
    d = df.copy()

    # created_utc: accetta sia timestamp (secondi) sia stringhe datetime
    if np.issubdtype(d["created_utc"].dtype, np.number):
        d["created_dt"] = pd.to_datetime(d["created_utc"], unit="s", utc=True, errors="coerce")
    else:
        d["created_dt"] = pd.to_datetime(d["created_utc"], utc=True, errors="coerce")

    # body words: conta parole in modo robusto (NaN -> 0)
    d["body"] = d["body"].fillna("").astype(str)
    d["body_word_count"] = d["body"].str.findall(r"\b\w+\b").str.len()

    # Normalizza type (minuscole e strip)
    d["type"] = d["type"].astype(str).str.strip().str.lower()

    # Aggregazioni per subreddit
    out = (
        d.groupby("subreddit_name", dropna=False)
         .agg(
            Total_Posts=("type", lambda s: (s == "post").sum()),
            Total_Comments=("type", lambda s: (s == "comment").sum()),
            Oldest_Date=("created_dt", "min"),
            Newest_Date=("created_dt", "max"),
            Avg_Words_per_Body=("body_word_count", "mean"),
         )
         .reset_index()
    )

    # Arrotonda media parole (opzionale)
    out["Avg_Words_per_Body"] = out["Avg_Words_per_Body"].round(2)

    # Se vuoi date senza timezone, decommenta:
    # out["Oldest_Date"] = out["Oldest_Date"].dt.tz_convert(None)
    # out["Newest_Date"] = out["Newest_Date"].dt.tz_convert(None)

    return out


In [ ]:
stats = subreddit_stats(conspiracy)
print(stats.sort_values("subreddit_name").to_string(index=False))

In [ ]:
stats = subreddit_stats(non_conspiracy)
print(stats.sort_values("subreddit_name").to_string(index=False))

In [ ]:
import pandas as pd

In [ ]:
conspiracy = pd.read_csv("consp2vec_dataset/conspiracy.csv")
non_conspiracy = pd.read_csv("consp2vec_dataset/non_conspiracy.csv")
emotions_conspiracy = pd.read_csv("consp2vec_dataset/emotions_conspiracy.csv")
emotions_non_conspiracy = pd.read_csv("consp2vec_dataset/emotions_non_conspiracy.csv")

In [ ]:
conspiracy

In [ ]:
total_post = 0
total_comment = 0

for g in set(conspiracy.subreddit_name):
    num_post = len(conspiracy[
    (conspiracy["subreddit_name"] == g) &
    (conspiracy["type"] == "post")])
    num_comment = len(conspiracy[
    (conspiracy["subreddit_name"] == g) &
    (conspiracy["type"] == "comment")])
    print(g+" / "+str(num_post)+" / "+str(num_comment))

    total_post += num_post
    total_comment += num_comment

print("Global"+" / "+str(total_post)+" / "+str(total_comment))

In [ ]:
total_post = 0
total_comment = 0
for g in set(non_conspiracy.subreddit_name):
    num_post = len(non_conspiracy[
    (non_conspiracy["subreddit_name"] == g) &
    (non_conspiracy["type"] == "post")])
    num_comment = len(non_conspiracy[
    (non_conspiracy["subreddit_name"] == g) &
    (non_conspiracy["type"] == "comment")])
    print(g+" / "+str(num_post)+" / "+str(num_comment))
    total_post += num_post
    total_comment += num_comment

print("Global"+" / "+str(total_post)+" / "+str(total_comment))

In [ ]:
import spacy
from spacytextblob.spacytextblob import SpacyTextBlob
import pandas as pd

# Carica il modello di SpaCy
nlp = spacy.load('en_core_web_sm')
nlp.add_pipe('spacytextblob')

def get_sentiment(row):
    # Gestisci i valori NaN nei titoli e nei corpi
    title = row["title"] if pd.notna(row["title"]) else ""
    body = row["body"] if pd.notna(row["body"]) else ""

    # Combina il titolo e il corpo
    text = title + " " + body

    # Analizza il testo con SpaCy
    doc = nlp(text)

    # Estrai polarità e soggettività
    return doc._.blob.polarity, doc._.blob.subjectivity


In [ ]:
from tqdm import tqdm

print(len(conspiracy))
conspiracy_sentiment = 0
conspiracy_polarity = 0
for row in tqdm(conspiracy.iterrows()):
    s,p = get_sentiment(row[1])
    conspiracy_sentiment + s
    conspiracy_polarity + p


print(conspiracy_sentiment/len(conspiracy))

